In [57]:
import sys
sys.path.append("..")
import pandas as pd
from src.data_processing import load_raw_tables

tables = load_raw_tables("../data/raw")

In [39]:
keep = {
 "P_DEMO": ["SEQN", "RIDSTATR", "RIDAGEYR", "RIAGENDR", "RIDRETH3", "RIDEXPRG",
           "INDFMPIR", "WTMECPRP", "SDMVPSU", "SDMVSTRA"],
"P_DIQ":["SEQN","DIQ010","DIQ050","DIQ070","DIQ160"],
 "P_GHB":["SEQN","LBXGH"],
 "P_BIOPRO":["SEQN","LBXSCR"],
 "P_ALB_CR":["SEQN","URXUMA","URXUCR","URDACT"],
 "P_BMX":["SEQN","BMXBMI","BMXWAIST"],
 "P_BPXO":["SEQN","BPXOSY1","BPXODI1"],
 "P_SMQ":["SEQN","SMQ020","SMQ040"]
}
merged = tables["P_DEMO"][keep["P_DEMO"]].copy()
for name in list(keep)[1:]:
    merged = merged.merge(tables[name][keep[name]], on="SEQN",
                          how="left", validate="one_to_one")
print(merged.shape)
assert len(merged) == 15560

(15560, 25)


In [40]:
# Identify adults aged 20 years or older
adults = merged[merged.RIDAGEYR >= 20].copy()

# Diabetes identification criteria
adults["dm_self"] = adults["DIQ010"] == 1
adults["dm_a1c"] = adults["LBXGH"] >= 6.5

# Primary diabetes definition:
# Self-reported diabetes OR HbA1c >= 6.5%
adults["diabetes"] = adults["dm_self"] | adults["dm_a1c"]

# Flag oral medication users reporting prediabetes
# without a diabetes diagnosis
adults["oral_med_prediabetes_only"] = (
    (adults["DIQ010"] == 2) &
    (adults["DIQ160"] == 1) &
    (adults["DIQ070"] == 1)
)

# Summary counts
print("Adults:", len(adults))
print("Self-reported diabetes:", adults["dm_self"].sum())
print("HbA1c >= 6.5%:", adults["dm_a1c"].sum())
print("Combined diabetes cohort:", adults["diabetes"].sum())

# Participants meeting each criterion
only_self = (
    adults["dm_self"] & ~adults["dm_a1c"]
).sum()

only_a1c = (
    adults["dm_a1c"] & ~adults["dm_self"]
).sum()

both = (
    adults["dm_self"] & adults["dm_a1c"]
).sum()

print("Only self-reported:", only_self)
print("Only HbA1c:", only_a1c)
print("Both criteria:", both)

# Medication-related descriptive information
pills_no_dx = adults[
    (adults["DIQ070"] == 1) &
    (adults["DIQ010"] != 1)
]

print(
    "Pills without diabetes diagnosis:", len(pills_no_dx),
    "| HbA1c >= 6.5%:", pills_no_dx["dm_a1c"].sum()
)

print(
    "Oral medication users reporting prediabetes "
    "without diabetes diagnosis:",
    adults["oral_med_prediabetes_only"].sum()
)

Adults: 9232
Self-reported diabetes: 1420
HbA1c >= 6.5%: 1171
Combined diabetes cohort: 1704
Only self-reported: 533
Only HbA1c: 284
Both criteria: 887
Pills without diabetes diagnosis: 108 | HbA1c >= 6.5%: 47
Oral medication users reporting prediabetes without diabetes diagnosis: 52


In [41]:
print(pd.crosstab(adults.DIQ010, adults.DIQ070, dropna=False))

DIQ070   1.0  2.0  9.0   NaN
DIQ010                      
1.0     1016  401    3     0
2.0       52  859    1  6633
3.0       56  205    1     0
9.0        0    2    0     3


In [44]:
# Diabetes cohort among adults
dm = adults[adults.diabetes].copy()

print("Diabetes cohort:", len(dm))

# Pregnancy status
print("\nPregnancy codes:")
print(dm.RIDEXPRG.value_counts(dropna=False))

# Adults without self-reported diabetes and with missing HbA1c
unknown_status = (
    (~adults.dm_self) &
    adults.LBXGH.isna()
)

print(
    "\nUnknown status (no self-reported diabetes, HbA1c missing):",
    unknown_status.sum()
)

# Missing HbA1c and examination weight
print("\nCohort HbA1c missing:", dm.LBXGH.isna().sum())

print(
    "Cohort WTMECPRP missing:",
    dm.WTMECPRP.isna().sum(),
    "| equal to 0:",
    (dm.WTMECPRP == 0).sum()
)

Diabetes cohort: 1704

Pregnancy codes:
RIDEXPRG
NaN    1607
2.0      84
3.0      10
1.0       3
Name: count, dtype: int64

Unknown status (no self-reported diabetes, HbA1c missing): 991

Cohort HbA1c missing: 160
Cohort WTMECPRP missing: 0 | equal to 0: 96


In [45]:
# Check whether RIDSTATR=2 matches WTMECPRP > 0
print(pd.crosstab(
    adults.RIDSTATR,
    adults.WTMECPRP > 0,
    dropna=False
))

# Restrict to examined adults
mec = adults[adults.RIDSTATR == 2].copy()

# Identify the updated diabetes cohort
dm_mec = mec[mec.diabetes].copy()

print("\nExamined adults:", len(mec))
print("Diabetes cohort:", len(dm_mec))

# Check missing HbA1c
print("Cohort HbA1c missing:", dm_mec.LBXGH.isna().sum())

# Check unknown diabetes status
unknown_status = (
    ~mec.dm_self &
    mec.LBXGH.isna()
)

print("Unknown status:", unknown_status.sum())

# Check pregnancy status
print("\nPregnancy status:")
print(dm_mec.RIDEXPRG.value_counts(dropna=False))

WTMECPRP  False  True 
RIDSTATR              
1.0         688      0
2.0           0   8544

Examined adults: 8544
Diabetes cohort: 1608
Cohort HbA1c missing: 64
Unknown status: 399

Pregnancy status:
RIDEXPRG
NaN    1517
2.0      84
3.0       4
1.0       3
Name: count, dtype: int64


cohort attrition table and outcome availability

In [46]:
# Final cohort: examined adults with diabetes,
# excluding pregnant and pregnancy-status-unascertained participants

cohort = dm_mec[
    ~dm_mec.RIDEXPRG.isin([1, 3])
].copy()

print("Final cohort:", len(cohort))

# Cohort attrition
flow = pd.DataFrame({
    "step": [
        "All participants",
        "Adults >=20",
        "Examined adults",
        "Diabetes (self-reported OR HbA1c >= 6.5%)",
        "Excluding pregnant/unascertained"
    ],
    "n": [
        len(merged),
        len(adults),
        len(mec),
        len(dm_mec),
        len(cohort)
    ]
})

display(flow)

# Kidney outcome availability
print("Creatinine missing:", cohort.LBXSCR.isna().sum())
print("UACR missing:", cohort.URDACT.isna().sum())
print(
    "Both missing:",
    (cohort.LBXSCR.isna() & cohort.URDACT.isna()).sum()
)
print("UACR >= 30:", (cohort.URDACT >= 30).sum())

Final cohort: 1601


,step,n
0,All participants,15560
1,Adults >=20,9232
2,Examined adults,8544
3,Diabetes (self-reported OR HbA1c >= 6.5%),1608
4,Excluding pregnant/unascertained,1601


Creatinine missing: 114
UACR missing: 56
Both missing: 15
UACR >= 30: 519


In [47]:
import numpy as np
from src.outcome_definition import egfr_ckdepi_2021, label_outcome

# Calculate kidney outcome using the updated cohort
labeled = label_outcome(cohort)

# Check kidney marker distribution
print("Kidney marker distribution:")
print(labeled.kidney_marker.value_counts(dropna=False))

# eGFR summary
print("\neGFR summary:")
print(labeled.egfr.describe())

# Kidney abnormality counts
print("\neGFR < 60:", (labeled.egfr < 60).sum())
print("UACR >= 30:", (labeled.URDACT >= 30).sum())

Kidney marker distribution:
kidney_marker
0.0    855
1.0    651
NaN     95
Name: count, dtype: int64

eGFR summary:
count    1487.000000
mean       83.507385
std        25.319714
min         2.958234
25%        67.338418
50%        87.786610
75%       102.486037
max       145.145698
Name: egfr, dtype: float64

eGFR < 60: 271
UACR >= 30: 519


In [48]:
from src.outcome_definition import egfr_ckdepi_2021, label_outcome

print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([False])))
print(egfr_ckdepi_2021(np.array([1.0]), np.array([50]), np.array([True])))

labeled = label_outcome(cohort)
print(labeled.kidney_marker.value_counts(dropna=False))
print(labeled.egfr.describe())
print("eGFR < 60:", (labeled.egfr < 60).sum())

[91.69147861]
[68.63349666]
kidney_marker
0.0    855
1.0    651
NaN     95
Name: count, dtype: int64
count    1487.000000
mean       83.507385
std        25.319714
min         2.958234
25%        67.338418
50%        87.786610
75%       102.486037
max       145.145698
Name: egfr, dtype: float64
eGFR < 60: 271


In [49]:
# Retain participants with an available kidney outcome
model_df = labeled[
    labeled.kidney_marker.notna()
].copy()

print("Modelling dataset shape:", model_df.shape)
print("\nKidney marker distribution:")
print(model_df.kidney_marker.value_counts())

# Sensitivity-analysis flag
model_df["pills_no_dx"] = (
    (model_df.DIQ070 == 1) &
    (model_df.DIQ010 != 1)
)

print(
    "\nPills without diabetes diagnosis:",
    model_df.pills_no_dx.sum()
)

# Save updated dataset
model_df.to_csv(
    "../data/interim/cohort_labeled.csv",
    index=False
)

Modelling dataset shape: (1506, 31)

Kidney marker distribution:
kidney_marker
0.0    855
1.0    651
Name: count, dtype: int64

Pills without diabetes diagnosis: 45


In [50]:
# Lowest eGFR rows: are they real?
print(labeled.sort_values("egfr")[["RIDAGEYR","RIAGENDR","LBXSCR","egfr","URDACT"]].head(8))
print("Max creatinine:", labeled.LBXSCR.max())

# Reasons for NaN labels
nan = labeled[labeled.kidney_marker.isna()]
print("Unlabeled:", len(nan))
print("Both missing:", (nan.egfr.isna() & nan.URDACT.isna()).sum())
print("Only creatinine missing (UACR normal):", (nan.egfr.isna() & nan.URDACT.notna()).sum())
print("Only UACR missing (eGFR normal):", (nan.egfr.notna() & nan.URDACT.isna()).sum())

       RIDAGEYR  RIAGENDR  LBXSCR      egfr   URDACT
11393      80.0       1.0   14.97  2.958234  1172.26
5749       72.0       2.0   10.96  3.383306      NaN
742        45.0       2.0   10.60  4.165567  3903.57
9136       66.0       1.0   10.78  4.785974  1555.26
15454      51.0       2.0    7.97  5.650477      NaN
13729      45.0       1.0    8.36  7.399229  5897.81
1359       66.0       1.0    7.43  7.480425      NaN
4568       80.0       1.0    6.75  7.693682  1561.18
Max creatinine: 14.97
Unlabeled: 95
Both missing: 15
Only creatinine missing (UACR normal): 63
Only UACR missing (eGFR normal): 17


In [58]:
# Keep only labelable people for modeling; keep the full labeled table for the report
model_df = labeled[labeled.kidney_marker.notna()].copy()
print(model_df.shape, model_df.kidney_marker.value_counts())

# Flag the sensitivity-analysis group: pills without a diabetes diagnosis
model_df["pills_no_dx"] = (model_df.DIQ070 == 1) & (model_df.DIQ010 != 1)
print("Pills-no-dx in modeling sample:", model_df.pills_no_dx.sum())

model_df.to_csv("../data/interim/cohort_labeled.csv", index=False)

(1506, 31) kidney_marker
0.0    855
1.0    651
Name: count, dtype: int64
Pills-no-dx in modeling sample: 45


In [52]:
pregnancy_check = adults[
    (adults.RIAGENDR == 2) &
    (adults.RIDAGEYR.between(20, 44))
]

print("Women aged 20–44:", len(pregnancy_check))

print("\nPregnancy status:")
print(pregnancy_check["RIDEXPRG"].value_counts(dropna=False))

print("\nMissing pregnancy status:")
print(pregnancy_check["RIDEXPRG"].isna().sum())

Women aged 20–44: 1874

Pregnancy status:
RIDEXPRG
2.0    1604
3.0     183
1.0      87
Name: count, dtype: int64

Missing pregnancy status:
0


In [53]:
print("Diabetes diagnosis vs insulin medication:")
print(pd.crosstab(
    adults["DIQ010"],
    adults["DIQ050"],
    dropna=False
))

print("\nDiabetes diagnosis vs oral medication:")
print(pd.crosstab(
    adults["DIQ010"],
    adults["DIQ070"],
    dropna=False
))

Diabetes diagnosis vs insulin medication:
DIQ050  1.0   2.0  9.0   NaN
DIQ010                      
1.0     413  1006    1     0
2.0       0     0    0  7545
3.0       0     0    0   262
9.0       0     0    0     5

Diabetes diagnosis vs oral medication:
DIQ070   1.0  2.0  9.0   NaN
DIQ010                      
1.0     1016  401    3     0
2.0       52  859    1  6633
3.0       56  205    1     0
9.0        0    2    0     3


In [54]:
print("Diabetes diagnosis vs prediabetes:")
print(pd.crosstab(
    adults["DIQ010"],
    adults["DIQ160"],
    dropna=False
))

print("\nOral medication users without diagnosed diabetes:")
print(
    adults.loc[
        (adults["DIQ010"] == 2) &
        (adults["DIQ070"] == 1),
        "DIQ160"
    ].value_counts(dropna=False)
)

Diabetes diagnosis vs prediabetes:
DIQ160  1.0   2.0  9.0   NaN
DIQ010                      
1.0       0     0    0  1420
2.0     912  6621   12     0
3.0       0     0    0   262
9.0       2     2    1     0

Oral medication users without diagnosed diabetes:
DIQ160
1.0    52
Name: count, dtype: int64


In [55]:
check = pd.read_csv("../data/interim/cohort_labeled.csv")

print("Saved dataset shape:", check.shape)
print("Missing target values:", check.kidney_marker.isna().sum())
print("Duplicate participant IDs:", check.SEQN.duplicated().sum())

Saved dataset shape: (1506, 32)
Missing target values: 0
Duplicate participant IDs: 0


In [56]:
print("In-memory columns:")
print(model_df.columns.tolist())

print("\nSaved CSV columns:")
print(check.columns.tolist())

print("\nUnexpected columns:")
print(set(check.columns) - set(model_df.columns))

In-memory columns:
['SEQN', 'RIDSTATR', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'RIDEXPRG', 'INDFMPIR', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'DIQ010', 'DIQ050', 'DIQ070', 'DIQ160', 'LBXGH', 'LBXSCR', 'URXUMA', 'URXUCR', 'URDACT', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXODI1', 'SMQ020', 'SMQ040', 'dm_self', 'dm_a1c', 'diabetes', 'oral_med_prediabetes_only', 'egfr', 'kidney_marker', 'pills_no_dx']

Saved CSV columns:
['SEQN', 'RIDSTATR', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'RIDEXPRG', 'INDFMPIR', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'DIQ010', 'DIQ050', 'DIQ070', 'DIQ160', 'LBXGH', 'LBXSCR', 'URXUMA', 'URXUCR', 'URDACT', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXODI1', 'SMQ020', 'SMQ040', 'dm_self', 'dm_a1c', 'diabetes', 'oral_med_prediabetes_only', 'egfr', 'kidney_marker', 'pills_no_dx']

Unexpected columns:
set()


In [59]:
print("In-memory columns:", len(model_df.columns))
print("Saved CSV columns:", len(pd.read_csv("../data/interim/cohort_labeled.csv").columns))

print("Columns:")
print(model_df.columns.tolist())

In-memory columns: 32
Saved CSV columns: 32
Columns:
['SEQN', 'RIDSTATR', 'RIDAGEYR', 'RIAGENDR', 'RIDRETH3', 'RIDEXPRG', 'INDFMPIR', 'WTMECPRP', 'SDMVPSU', 'SDMVSTRA', 'DIQ010', 'DIQ050', 'DIQ070', 'DIQ160', 'LBXGH', 'LBXSCR', 'URXUMA', 'URXUCR', 'URDACT', 'BMXBMI', 'BMXWAIST', 'BPXOSY1', 'BPXODI1', 'SMQ020', 'SMQ040', 'dm_self', 'dm_a1c', 'diabetes', 'oral_med_prediabetes_only', 'egfr', 'kidney_marker', 'pills_no_dx']
